<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Module 0 · Foundations</h4>
<h1 align="center">Plotly Essentials</h1>
<p align="center"><i>Interactive, presentation-ready charts with plotly.express and graph_objects</i></p>

---

## About This Notebook

This is the Plotly section of the Module 0 refresher, split out so it can be run on its own,
in any order relative to the other Module 0 notebooks. See `00_numpy_and_pandas.ipynb` for
the NumPy/pandas fundamentals, and `01_matplotlib.ipynb` / `02_seaborn.ipynb` for the other
two visualization libraries.

## 0. Recreating the Dataset from Notebook 0

This notebook is meant to run on its own, in any order relative to the other Module 0
notebooks. So before we get to plotting, we rebuild the exact same synthetic "store sales"
DataFrame introduced in `00_numpy_and_pandas.ipynb`, using the same random seed (`42`), so the
data -- and every chart built from it -- is identical no matter which notebook you open
first.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

regions = rng.choice(["North", "South", "East", "West"], size=n, p=[0.3, 0.3, 0.2, 0.2])
categories = rng.choice(["Electronics", "Grocery", "Apparel"], size=n)
units_sold = rng.poisson(lam=20, size=n)
unit_price = np.round(rng.uniform(5, 200, size=n), 2)
customer_rating = np.clip(rng.normal(loc=4.0, scale=0.7, size=n), 1, 5).round(1)

sales = pd.DataFrame({
    "region": regions,
    "category": categories,
    "units_sold": units_sold,
    "unit_price": unit_price,
    "revenue": np.round(units_sold * unit_price, 2),
    "customer_rating": customer_rating,
})

# Inject a few realistic missing values, so plots involving customer_rating have gaps to handle
missing_idx = rng.choice(sales.index, size=15, replace=False)
sales.loc[missing_idx, "customer_rating"] = np.nan

print(sales.shape)
sales.head()

(300, 6)


,region,category,units_sold,unit_price,revenue,customer_rating
0,East,Grocery,23,74.69,1717.87,3.9
1,South,Apparel,13,56.64,736.32,4.3
2,West,Apparel,24,124.04,2976.96,2.5
3,East,Electronics,21,45.05,946.05,4.0
4,North,Apparel,29,178.23,5168.67,5.0


### 0.1 Shared color palette

We reuse the same categorical/region color mapping as the other Module 0 notebooks, this time
passed to Plotly as `color_discrete_map`.

In [2]:
BLUE, RED, GREEN, ORANGE, PURPLE = "#4C72B0", "#C44E52", "#55A868", "#DD8452", "#8172B2"

## 1. Plotly Essentials

Plotly's `plotly.express` module (imported as `px`) builds full interactive charts --
zoomable, pannable, with hover tooltips -- in one function call. Reach for Plotly whenever
the *interactivity itself* (hovering to inspect a point, zooming into a crowded region,
rotating a 3D view) adds value, typically during live exploration or in a presentation;
reach for Matplotlib/Seaborn instead when you need a static image for a paper, report, or
print.

In [3]:
import plotly.express as px

fig = px.scatter(
    sales, x="unit_price", y="units_sold", color="category", size="revenue",
    hover_data=["region", "customer_rating"],
    color_discrete_map={"Electronics": BLUE, "Grocery": GREEN, "Apparel": ORANGE},
    title="px.scatter: price vs. units sold (hover for details, drag to zoom)",
)
fig.update_layout(height=450)
fig.show()

In [4]:
region_summary = sales.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    avg_rating=("customer_rating", "mean"),
    n_orders=("revenue", "count"),
).sort_values("total_revenue", ascending=False)

fig = px.bar(
    region_summary.reset_index(), x="region", y="total_revenue", color="region",
    color_discrete_map={"North": BLUE, "South": RED, "East": GREEN, "West": ORANGE},
    title="px.bar: total revenue by region",
)
fig.update_layout(height=400, showlegend=False)
fig.show()

In [5]:
daily_by_category = sales.groupby([sales.index // 10, "category"])["revenue"].sum().reset_index()
daily_by_category.columns = ["order_bucket", "category", "revenue"]

fig = px.line(
    daily_by_category, x="order_bucket", y="revenue", color="category",
    color_discrete_map={"Electronics": BLUE, "Grocery": GREEN, "Apparel": ORANGE},
    title="px.line: revenue per order-bucket, by category",
)
fig.update_layout(height=400)
fig.show()

In [6]:
fig = px.histogram(
    sales, x="customer_rating", color="category", barmode="overlay", opacity=0.6,
    color_discrete_map={"Electronics": BLUE, "Grocery": GREEN, "Apparel": ORANGE},
    title="px.histogram: customer rating distribution by category",
)
fig.update_layout(height=400)
fig.show()

### A multivariate view: `scatter_3d`

Plotly's `scatter_3d` lets you rotate and inspect a 3D relationship interactively --
something a static Matplotlib 3D plot can only approximate with one fixed viewing angle
per image.

In [7]:
fig = px.scatter_3d(
    sales, x="unit_price", y="units_sold", z="revenue", color="category",
    color_discrete_map={"Electronics": BLUE, "Grocery": GREEN, "Apparel": ORANGE},
    opacity=0.7, title="px.scatter_3d: price, units sold, and revenue at once",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=550)
fig.show()

### Building a figure by hand with `graph_objects`

`plotly.express` covers most everyday charts in one line. When you need to combine several
custom traces into one figure -- e.g. overlaying two different chart types on shared axes --
drop down to `plotly.graph_objects` (`go`), which is what `px` itself generates under the
hood.

In [8]:
import plotly.graph_objects as go

by_region = sales.groupby("region").agg(total_revenue=("revenue", "sum"), avg_rating=("customer_rating", "mean")).reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(x=by_region["region"], y=by_region["total_revenue"], name="Total revenue", marker_color=BLUE, yaxis="y1"))
fig.add_trace(go.Scatter(x=by_region["region"], y=by_region["avg_rating"], name="Avg. rating", mode="lines+markers", marker_color=RED, yaxis="y2"))

fig.update_layout(
    title="go.Figure: combining a bar trace and a line trace on two y-axes",
    yaxis=dict(title="total revenue"),
    yaxis2=dict(title="avg. rating", overlaying="y", side="right", range=[0, 5]),
    height=450, legend=dict(orientation="h", y=-0.2),
)
fig.show()

## 2. Stunning, Presentation-Ready Visuals

Everything so far used our small synthetic sales table. To show off what Plotly can really do
-- the kind of chart worth putting in front of an audience -- we borrow `px.data.gapminder()`,
a real, historical dataset bundled with Plotly itself (life expectancy, population, and GDP
per capita for 142 countries, 1952-2007), alongside one purely mathematical example. No
internet connection or extra download is needed for any of this.

### 2.1 A true 3D surface with `go.Surface`

`scatter_3d` (Section 1) plots individual points in 3D. `go.Surface` instead plots a
continuous function of two variables as a shaded, rotatable sheet -- the kind of visual that
makes an optimization landscape or a physical simulation genuinely intuitive to explore.

In [9]:
xs = np.linspace(-5, 5, 80)
ys = np.linspace(-5, 5, 80)
XS, YS = np.meshgrid(xs, ys)
R = np.sqrt(XS**2 + YS**2)
ZS = np.sin(R) / (R + 0.001)  # a rippling "sombrero" function -- purely to show off the surface

fig = go.Figure(data=[go.Surface(x=xs, y=ys, z=ZS, colorscale="Viridis")])
fig.update_layout(
    title="go.Surface: a continuous function, fully rotatable",
    height=520, scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
)
fig.show()

### 2.2 The classic animated bubble chart

Passing `animation_frame` to almost any `px` chart turns it into a play-button animation with
a scrubber slider, at no extra cost beyond that one argument. This is the famous
Gapminder/Hans Rosling chart: each bubble is a country, sized by population, moving through
time as the world's wealth and health both rise.

In [10]:
gapminder = px.data.gapminder()  # bundled with plotly -- no download needed

fig = px.scatter(
    gapminder, x="gdpPercap", y="lifeExp", size="pop", color="continent", hover_name="country",
    animation_frame="year", animation_group="country", log_x=True, size_max=55,
    range_x=[100, 100_000], range_y=[20, 90],
    title="px.scatter animated: GDP per capita vs. life expectancy, 1952-2007",
)
fig.update_layout(height=550)
fig.show()

### 2.3 Regional data on an actual map with `px.choropleth`

Whenever your rows have a country, state, or other region identifier, `px.choropleth` colors
an actual map by a numeric column -- exactly the right tool any time "regional data" needs a
visual, rather than forcing it into a bar chart.

In [11]:
fig = px.choropleth(
    gapminder, locations="iso_alpha", color="lifeExp", hover_name="country",
    animation_frame="year", color_continuous_scale=px.colors.sequential.Plasma,
    range_color=[20, 90], title="px.choropleth animated: life expectancy by country, 1952-2007",
)
fig.update_layout(height=550)
fig.show()

### 2.4 Hierarchical data with `px.sunburst`

Back to our own sales data: whenever rows nest into categories-within-categories (region, then
category, within it), a sunburst shows the whole hierarchy and each level's relative share in
one glance -- something a bar chart can't do without picking one grouping level and dropping
the other.

In [12]:
fig = px.sunburst(
    sales, path=["region", "category"], values="revenue", color="region",
    color_discrete_map={"North": BLUE, "South": RED, "East": GREEN, "West": ORANGE},
    title="px.sunburst: revenue by region, then category within region",
)
fig.update_layout(height=550)
fig.show()

**The pattern across all four:** none of this required a new charting library or a design
tool -- `go.Surface`, `animation_frame`, `px.choropleth`, and `px.sunburst` are each a single
argument or function swap away from the basic `px.scatter`/`px.bar` calls in Section 1. The
"stunning" part is Plotly's default styling and interactivity doing the work; the code stays
almost as short as the simple charts it started from.

## 3. Which Tool, When?

The original Module 0 decision guide is split across this folder's four notebooks, one entry
per notebook. Here is the entry for Plotly:

- **Plotly** -- reach for it when interactivity earns its keep: hovering for exact values,
  zooming into a crowded region, rotating a 3D scatter, animating through time, or building a
  chart for a live demo or dashboard rather than a static document.

See `00_numpy_and_pandas.ipynb`, `01_matplotlib.ipynb`, and `02_seaborn.ipynb` for the other
entries -- together the four notebooks in this folder cover the full NumPy / pandas /
Matplotlib / Seaborn / Plotly decision guide.

## Summary

- Rebuilt the same synthetic "store sales" dataset used across every Module 0 notebook.
- **Plotly essentials**: quick interactive charts via `plotly.express` (`scatter`, `bar`,
  `line`, `histogram`, `scatter_3d`), plus hand-building a multi-trace figure with
  `plotly.graph_objects`.
- **Stunning, presentation-ready visuals**: a rotatable 3D `go.Surface`, the classic animated
  Gapminder bubble chart (`animation_frame`), a `px.choropleth` map for regional data, and a
  `px.sunburst` for hierarchical data -- all one argument or function swap away from the
  basic charts above.
- When Plotly is the right call vs. reaching for Matplotlib or Seaborn instead.

This is the last notebook in the Module 0 refresher -- see `00_numpy_and_pandas.ipynb`,
`01_matplotlib.ipynb`, and `02_seaborn.ipynb` for the rest of it.